# An independent simulator: SCOPE (Supplementary Note 1)

The SCOPE assay (Liao et al.) uses arrays in which every bead both sends and receives barcodes, and its authors released their own simulator. Benchmarking on it is a check that does not depend on our calibrated simulation. This notebook reproduces the SCOPE results in Supplementary Note 1 and Supp Table S2:
1. accumap against UMAP and similarity-weighted multidimensional scaling on 5 simulated arrays;
2. why the contact-isotropy prior needs its array-wide-mean option here: SCOPE's simulator transfers more barcodes along lattice rows than diagonally, which accumap's built-in check detects;
3. recovering an array that UMAP embedded with a mirrored region, using several UMAP starts.

**The simulator** is SCOPE's `simulation.py` (github.com/SrivatsanLab/SCOPE), run in its published configuration: a 197 × 197 hexagonal close-packed grid (38,809 beads; units are bead diameters), inverse-square transfer truncated at 50 bead diameters, per-bead totals resampled from SCOPE's experimental data (`paper_results/scope/joint_sums.csv`), multinomial counts. The SCOPE repository has no license, so its file is not included here; with `RECOMPUTE = True` it is downloaded.

**Runtime.** By default (`RECOMPUTE = False`) everything is drawn from `paper_results/` in seconds. `RECOMPUTE = True` regenerates and reconstructs all 5 arrays (about 2 hours on one GPU, mostly UMAP on 38,809 beads; the similarity MDS baseline needs about 25 GB of RAM and the baseline's Rust solver, see `baselines/__init__.py`) and writes to `paper_results/scope_recomputed/`.

In [ ]:
import os, re, json, colorsys
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
from accumap.metrics import evaluate, similarity_align

RECOMPUTE = False                      # True: regenerate and reconstruct all 5 arrays (GPU, about 2 h)
RESULTS = "paper_results"
OUTDIR = os.path.join(RESULTS, "scope_recomputed") if RECOMPUTE else os.path.join(RESULTS, "scope")
os.makedirs(OUTDIR, exist_ok=True)
SEEDS = range(5)
parse = lambda m: [float(x) for x in re.findall(r"(?:p50|p90|pairRMSE|J@10) ([\d.]+)", m)]
fmt = lambda m: f"recv p50 {m['median']:.2f} p90 {m['p90']:.2f} pairRMSE {m['pairwise_rmse']:.2f} J@10 {m['jaccard10']:.3f}"

## 1. Simulating and reconstructing the arrays (`RECOMPUTE = True` only)

UMAP is run on the symmetrized counts (sent plus received). accumap runs in shared-bead mode with the array's rectangular outline as its density footprint, and with the isotropy prior measured relative to its array-wide mean (`iso_center=True`, section 2). The similarity MDS baseline is restricted to each bead's five most similar neighbors (the full objective is too slow at this size), with its similarity-to-distance curve calibrated on 100 beads with known positions.

In [ ]:
if RECOMPUTE:
    import urllib.request, importlib.util, umap, torch
    from accumap.count_model import reconstruct, rescale_to_diameter
    from baselines.curves import fit_curve, build_knn_mask, accumap_solve
    assert torch.cuda.is_available(), "RECOMPUTE needs a CUDA GPU"
    path = os.path.join(OUTDIR, "scope_simulation.py")
    urllib.request.urlretrieve("https://raw.githubusercontent.com/SrivatsanLab/SCOPE/main/simulation.py", path)
    spec = importlib.util.spec_from_file_location("scope_simulation", path); scope_sim = importlib.util.module_from_spec(spec); spec.loader.exec_module(scope_sim)
    js_all = pd.read_csv(os.path.join(RESULTS, "scope", "joint_sums.csv"))

    def simulate(seed):
        np.random.seed(seed)
        js = js_all.sample(frac=1.0, random_state=seed).reset_index(drop=True)
        sim = scope_sim.BaseSimulation(197, 197, max_dispersion_radius=50, max_dispersion_scale=50, joint_sums=js)
        counts = sim.simulate_experiment()
        xy = sim.add_coords(sim.bead_df)[["x_coord", "y_coord"]].values.astype(np.float64)
        C = sp.coo_matrix((counts["bead_counts"].values.astype(np.float32), (counts["source_bead"].values, counts["target_bead"].values)),
                          shape=(len(xy), len(xy))).tocsr()
        return C, (xy - xy.mean(0)).astype(np.float32), float(np.hypot(*np.ptp(xy, 0)))

    def umap_shared(C, seed):
        S = (C + C.T).tocsr().astype(np.float32); S.data = np.log1p(S.data)
        return umap.UMAP(random_state=seed, metric="cosine", n_neighbors=25, min_dist=0.99, n_epochs=400).fit_transform(S).astype(np.float32)

    def weighted_mds_knn(C, X, E, extent):
        S = (C + C.T).tocsr().astype(np.float64); S.data = np.log1p(S.data)
        S = sp.diags(1 / (np.sqrt(np.asarray(S.multiply(S).sum(1)).ravel()) + 1e-12)) @ S
        S = (S @ S.T).toarray(); np.fill_diagonal(S, 0.0)
        k = 3000.0 / extent                                       # the original method works in a 3000-unit frame
        cal = np.random.RandomState(42).randint(0, len(X), 100)
        Dt = np.zeros_like(S, dtype=np.float32); Dt[cal] = np.linalg.norm(X[cal][:, None] - X[None], axis=2) * k
        D = fit_curve(S, Dt); del Dt
        return accumap_solve(rescale_to_diameter(E, 3000.0), D, np.where(build_knn_mask(S, 5), S, 0.0), cycles=10)

    transfer = []
    for s in SEEDS:
        C, X, extent = simulate(s)
        lo, hi = X.min(0), X.max(0); rect = np.array([[lo[0], lo[1]], [hi[0], lo[1]], [hi[0], hi[1]], [lo[0], hi[1]]])
        E = umap_shared(C, s)
        R = reconstruct(C, E, extent, shared=True, footprint=rect, iso_center=True, verbose=False)
        rec = {"umap": E, "new": R["receivers"], "old_knn": weighted_mds_knn(C, X, E, extent)}
        json.dump({m: fmt(evaluate(P, X)) for m, P in rec.items()}, open(os.path.join(OUTDIR, f"scope_seed{s}.json"), "w"))
        transfer.append({"seed": s, **R["transfer_anisotropy"]})
        if s == 0:
            np.savez_compressed(os.path.join(OUTDIR, "seed0_example.npz"), X=X, **rec)
        if s == 1:   # UMAP mirrors a corner of this array; several starts recover it
            R3 = reconstruct(C, E, extent, shared=True, footprint=rect, iso_center=True, n_starts=3, verbose=False)
            json.dump({"n_starts3": fmt(evaluate(R3["receivers"], X))}, open(os.path.join(OUTDIR, "nstarts.json"), "w"))
        print(s, "done", flush=True)
    with open(os.path.join(OUTDIR, "transfer_check.jsonl"), "w") as fh:
        fh.writelines(json.dumps(r) + "\n" for r in transfer)

## Results (Supp Table S2, SCOPE rows)

Errors are in bead diameters. On a perfect hexagonal lattice the 10 nearest neighbors are not unique (ties), so even the true layout has a Jaccard distance of about 0.08.

In [ ]:
def scope_metrics(seed):
    if RECOMPUTE:
        return {m: parse(v) for m, v in json.load(open(os.path.join(OUTDIR, f"scope_seed{seed}.json"))).items()}
    sim = os.path.join(RESULTS, "simulations")
    final = json.load(open(os.path.join(sim, f"scope_seed{seed}_umap-v2.json")))
    old = json.load(open(os.path.join(sim, f"scope_seed{seed}_umap-old_knn.json")))
    return {"umap": parse(final["umap"]), "old_knn": parse(old["old_knn"]), "new": parse(final["new"])}

M = {s: scope_metrics(s) for s in SEEDS}
NAMES = {"umap": "UMAP", "old_knn": "similarity MDS (5-NN)", "new": "accumap"}
rows = []
for m, name in NAMES.items():
    A = np.array([M[s][m] for s in SEEDS])
    r = {"method": name}
    for i, k in enumerate(("median", "p90", "pairwise RMSE", "10-NN Jaccard")):
        r[k] = f"{A[:, i].mean():.3g} ± {A[:, i].std(ddof=1):.2g}"
    if m != "umap":
        U = np.array([M[s]["umap"] for s in SEEDS])
        r["arrays better than UMAP (median / J@10)"] = f"{int((A[:, 0] < U[:, 0]).sum())}/5, {int((A[:, 3] < U[:, 3]).sum())}/5"
    rows.append(r)
pd.DataFrame(rows).set_index("method")

In [ ]:
Z = np.load(os.path.join(OUTDIR, "seed0_example.npz"))
X = Z["X"]
h = 0.05 + 0.85 * (X[:, 0] - X[:, 0].min()) / np.ptp(X[:, 0]); l = 0.3 + 0.3 * (X[:, 1] - X[:, 1].min()) / np.ptp(X[:, 1])
cols = np.array([colorsys.hls_to_rgb(a, b, 0.65) for a, b in zip(h, l)])
fig, axes = plt.subplots(1, 4, figsize=(18, 4.8))
for ax, (name, key) in zip(axes, (("truth", None), ("UMAP", "umap"), ("similarity MDS (5-NN)", "old_knn"), ("accumap", "new"))):
    if key is None:
        P = X - X.mean(0)
    else:
        P, T = similarity_align(np.nan_to_num(Z[key]), X)
        name += f"\nmedian error {np.median(np.linalg.norm(P - T, axis=1)):.2f} bead diameters"
    ax.scatter(P[:, 0], P[:, 1], c=cols, s=1.2, linewidths=0, rasterized=True)
    ax.set_aspect("equal"); ax.axis("off"); ax.set_title(name)
plt.suptitle("SCOPE simulator, array 0 (beads colored by true position)", y=1.02); plt.show()

## 2. Direction-dependent transfer in SCOPE's simulator

accumap's contact-isotropy prior assumes that neither the bead arrangement nor barcode transfer has a preferred direction. SCOPE's beads lie on a regular hexagonal lattice, with every nearest neighbor at the same distance, but the simulator assigns about 20 percent more counts to neighbors along a lattice row than to diagonal neighbors. Contact directions at the true positions are therefore anisotropic (about 0.12), which the prior would wrongly treat as shear. Measuring anisotropy relative to its array-wide mean (`iso_center=True`) ignores such a uniform bias.

accumap detects this from the counts alone: after reconstruction it tests whether observed-to-expected counts over contacts depend on the contact's axis. Directional transfer changes the counts, whereas a shear of the layout does not. It warns when the z-score exceeds 4.

In [ ]:
if RECOMPUTE:
    from scipy.spatial import cKDTree
    db = {}
    for s in SEEDS:
        C, X, _ = simulate(s)
        dist, idx = cKDTree(X).query(X, 7); idx = idx[:, 1:]
        v = X[idx] - X[:, None]; ang = np.degrees(np.arctan2(v[..., 1], v[..., 0])) % 180
        S = (C + C.T).tocsr(); cnt = np.asarray(S[np.repeat(np.arange(len(X)), 6), idx.ravel()]).reshape(-1, 6)
        by = {n: float(cnt[(ang >= a) & (ang < b)].mean()) for n, a, b in (("along rows (0 deg)", 0, 15), ("diagonal 60 deg", 45, 75), ("diagonal 120 deg", 105, 135))}
        comp = []
        for i in range(len(X)):     # each bead's contacts: its 3 highest-count partners with >= 3 counts
            d_, ix = C.data[C.indptr[i]:C.indptr[i + 1]], C.indices[C.indptr[i]:C.indptr[i + 1]]
            o = np.argsort(d_)[::-1][:3]; top = ix[o][d_[o] >= 3]
            if len(top):
                u = X[top] - X[i]; u /= np.linalg.norm(u, axis=1, keepdims=True); comp += [[x * x - y * y, 2 * x * y] for x, y in u]
        db[s] = {"mean_count_to_nearest_neighbor": by, "anisotropy_at_truth": float(np.hypot(*np.mean(comp, 0)))}
    json.dump(db, open(os.path.join(OUTDIR, "direction_bias.json"), "w"), indent=1)

db = json.load(open(os.path.join(OUTDIR, "direction_bias.json")))
tc = {r["seed"]: r for r in map(json.loads, open(os.path.join(OUTDIR, "transfer_check.jsonl")))}
pd.DataFrame({int(s): {**d["mean_count_to_nearest_neighbor"], "contact anisotropy at truth": d["anisotropy_at_truth"],
                       "transfer check z": tc[int(s)]["z"], "favored axis (deg)": tc[int(s)]["axis_deg"]} for s, d in db.items()}).T.round(3)

## 3. A mirrored UMAP embedding, recovered with several starts

In array 1, UMAP embeds a corner region (about 5 percent of the beads) mirror-imaged. A smooth warp cannot undo a fold, and per-bead refinement cannot reach it, so the reconstruction keeps a displaced block. The fold is detectable without ground truth: the folded layout's maximized Poisson log-likelihood is lower by about 6.5 × 10^5 than that of layouts from other UMAP starts, which agree with each other to within 5 × 10^3. `reconstruct(..., n_starts=3)` runs from several UMAP initializations and keeps the most likely layout.

In [ ]:
one = parse(json.load(open(os.path.join(OUTDIR, "nstarts.json")))["n_starts3"])
pd.DataFrame({"single start (folded UMAP)": M[1]["new"], "n_starts = 3": one},
             index=["median", "p90", "pairwise RMSE", "10-NN Jaccard"]).T